In [1]:
import sys, pathlib
repo_root = pathlib.Path("/Users/muhammadnumanmuttaqi/Documents/MScITBE/Thesis/Thesis/Geometry-to-Ontology")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

### DOOR


In [ ]:
# CREATE INSTANCE DOOR IF MISSING
from rdflib import Graph
import pathlib

# Paths
core = pathlib.Path('../ontology/core/Resplan.ttl')
input_plan = pathlib.Path('../output/imp_resplan_ttl/plan_00010_drop_door.ttl')
construct_dir = pathlib.Path('../ontology/construct')
construct_files = [
    construct_dir / 'DoorConstruct.rq',
]
output_path = pathlib.Path('../output/inferred_resplan_ttl/plan_00010_doors_back.ttl')

# Merge data graph(s)
g = Graph()
for src in (core, input_plan):
    print(f'Loading: {src}')
    g.parse(src, format='turtle')
print(f'Loaded triples: {len(g)}')

# Apply CONSTRUCT queries (use g.query, not g.update)
for cq in construct_files:
    text = cq.read_text()
    print(f'Applying construct: {cq}')
    res = g.query(text)
    try:
        g += res.graph 
    except Exception:
        for t in res:
            g.add(t)
print(f'Triples after construct: {len(g)}')

# Save enriched graph
output_path.parent.mkdir(parents=True, exist_ok=True)
g.serialize(output_path, format='turtle')
print(f'Saved: {output_path}')


Loading: ../ontology/core/Resplan.ttl
Loading: ../output/imp_resplan_ttl/plan_00010_drop_door.ttl
Loaded triples: 1356
Applying construct: ../ontology/construct/DoorConstruct.rq
Triples after construct: 1365
Saved: ../output/inferred_resplan_ttl/plan_00010_doors_back.ttl


In [ ]:
# REVALIDATION: DOOR RULES MISSING CHECKER
from pyshacl import validate
from rdflib import RDF, Graph, Namespace, RDFS, BNode, Literal

def _short(node):
    return str(node).split("#")[-1] if "#" in str(node) else str(node)

data_graph = Graph().parse("../output/inferred_resplan_ttl/plan_00000_doors_back.ttl", format="turtle")
shacl_graph = Graph().parse("../ontology/rules/DoorRule.shacl.ttl", format="turtle")
onto_graph = Graph().parse("../ontology/core/Resplan.ttl", format="turtle")

conforms, report_graph, _ = validate(
    data_graph,
    shacl_graph=shacl_graph,
    ont_graph=onto_graph,
    inference="rdfs",
    meta_shacl=True,
    advanced=True,
)

sh = Namespace("http://www.w3.org/ns/shacl#")

if conforms:
    print("All door rules satisfied.")
else:
    print("Door rule violations found:")

    handled_sparql_rules = set()

    for result in report_graph.subjects(RDF.type, sh.ValidationResult):
        raw_shape = report_graph.value(result, sh.sourceShape)

        # mapped from PropertyShape/SPARQL bnode to main NodeShape (using shacl_graph!)
        rule_node = raw_shape
        if isinstance(raw_shape, BNode):
            for ns in shacl_graph.subjects(sh.property, raw_shape):
                rule_node = ns
                break

        rule_id = _short(rule_node)
        rule_label = report_graph.value(rule_node, RDFS.label) or shacl_graph.value(rule_node, RDFS.label)
        rule_name = str(rule_label) if rule_label else rule_id

        comp = report_graph.value(result, sh.sourceConstraintComponent)
        focus = _short(report_graph.value(result, sh.focusNode))
        message_node = report_graph.value(result, sh.resultMessage)
        message = str(message_node) if isinstance(message_node, Literal) or message_node else ""

        if comp == sh.SPARQLConstraintComponent:
            if rule_node in handled_sparql_rules:
                continue  # already processed this rule
            handled_sparql_rules.add(rule_node)
            
            sc = report_graph.value(result, sh.sourceConstraint)
            sparql = report_graph.value(sc, sh.select)

            for row in data_graph.query(str(sparql)):
                if len(row) >= 2:
                    print(f"{_short(row[0])} – {_short(row[1])}: {message} ({rule_name})")
                else:
                    print(f"{' – '.join(_short(v) for v in row)}: {message} ({rule_name})")
        else:
            print(f"- {focus} violates {rule_name}: {message}")


All door rules satisfied.


In [5]:
# Debugging
from rdflib import Namespace
g = Graph().parse('../output/inferred_resplan_ttl/plan_00000_doors_back.ttl', format='turtle')
resplan = Namespace('http://resplan.org/resplan#')

inferred_doors = [d for d in g.subjects(None, resplan.Door) if 'infer#' in str(d)]
for d in inferred_doors:
    spaces = list(g.objects(d, resplan.connectsSpace))
    print(str(d).split('#')[-1], '->', [str(s).split('#')[-1] for s in spaces])


11574d845d6e58e8d5315ec817a409e7e47f2e11478c9c5884c77d526e36cbc4 -> ['BED-01', 'LIV-01']


In [31]:
import requests

# Inferenced output path
ttl_path = pathlib.Path("../output/inferred_resplan_ttl/plan_00010_doors_back.ttl")
ttl_data = ttl_path.read_bytes()

# configure GraphDB
repo_id = "SemanticToGeometry_01"
endpoint = f"http://localhost:7200/repositories/{repo_id}/statements"

# named graph to save
graph_iri = "http://resplan.org/graph/data/plan_00010_doors_back"

# delete existing graph if available
clear_update = f"CLEAR GRAPH <{graph_iri}>"
requests.post(
    endpoint,
    data=clear_update,
    headers={"Content-Type": "application/sparql-update"}
).raise_for_status()

# Upload ttl to named graph
resp = requests.post(
    endpoint,
    params={"context": f"<{graph_iri}>"},
    data=ttl_data,
    headers={"Content-Type": "text/turtle"},
)
resp.raise_for_status()
print("Uploaded to GraphDB:", graph_iri)



Uploaded to GraphDB: http://resplan.org/graph/data/plan_00010_doors_back


### WINDOW

In [ ]:
# CREATE INSTANCE WINDOW IF MISSING
from rdflib import Graph
import pathlib

# Paths
core = pathlib.Path('../ontology/core/Resplan.ttl')
input_plan = pathlib.Path('../output/imp_resplan_ttl/plan_00010_drop_window.ttl')
construct_dir = pathlib.Path('../ontology/construct')
construct_files = [
    construct_dir / 'WindowConstruct.rq',
]
output_path = pathlib.Path('../output/inferred_resplan_ttl/plan_00010_windows_back.ttl')

# Merge data graph(s)
g = Graph()
for src in (core, input_plan):
    print(f'Loading: {src}')
    g.parse(src, format='turtle')
print(f'Loaded triples: {len(g)}')

# Apply CONSTRUCT queries (use g.query, not g.update)
for cq in construct_files:
    text = cq.read_text()
    print(f'Applying construct: {cq}')
    res = g.query(text)
    try:
        g += res.graph  
    except Exception:
        # If result is not a Graph (older rdflib), iterate bindings
        for t in res:
            g.add(t)
print(f'Triples after construct: {len(g)}')

# Save enriched graph
output_path.parent.mkdir(parents=True, exist_ok=True)
g.serialize(output_path, format='turtle')
print(f'Saved: {output_path}')


Loading: ../ontology/core/Resplan.ttl
Loading: ../output/imp_resplan_ttl/plan_00010_drop_window.ttl
Loaded triples: 1334
Applying construct: ../ontology/construct/WindowConstruct.rq
Triples after construct: 1364
Saved: ../output/inferred_resplan_ttl/plan_00010_windows_back.ttl


In [ ]:
# REVALIDATION: WINDOW RULES MISSING CHECKER
from pyshacl import validate
from rdflib import RDF, RDFS, BNode, Literal, Graph, Namespace

def _short(node):
    node = str(node)
    return node.split('#')[-1] if '#' in node else node

data_graph = Graph().parse('../output/inferred_resplan_ttl/plan_00000_windows_back.ttl', format='turtle')
shacl_graph = Graph().parse('../ontology/rules/WindowRule.shacl.ttl', format='turtle')
onto_graph = Graph().parse('../ontology/core/Resplan.ttl', format='turtle')

conforms, report_graph, _ = validate(
    data_graph,
    shacl_graph=shacl_graph,
    ont_graph=onto_graph,
    inference="rdfs",
    meta_shacl=True,
    advanced=True,
)

sh = Namespace("http://www.w3.org/ns/shacl#")

if conforms:
    print("All window rules satisfied.")
else:
    print("Window rule violations found:")

    handled_sparql_rules = set()

    for result in report_graph.subjects(RDF.type, sh.ValidationResult):
        raw_shape = report_graph.value(result, sh.sourceShape)

        # mapped from PropertyShape/SPARQL bnode to main NodeShape (using shacl_graph!)
        rule_node = raw_shape
        if isinstance(raw_shape, BNode):
            for ns in shacl_graph.subjects(sh.property, raw_shape):
                rule_node = ns
                break

        rule_id = _short(rule_node)
        rule_label = report_graph.value(rule_node, RDFS.label) or shacl_graph.value(rule_node, RDFS.label)
        rule_name = str(rule_label) if rule_label else rule_id

        comp = report_graph.value(result, sh.sourceConstraintComponent)
        focus = _short(report_graph.value(result, sh.focusNode))
        message_node = report_graph.value(result, sh.resultMessage)
        message = str(message_node) if isinstance(message_node, Literal) or message_node else ""

        if comp == sh.SPARQLConstraintComponent:
            if rule_node in handled_sparql_rules:
                continue  # already processed this rule
            handled_sparql_rules.add(rule_node)
            
            sc = report_graph.value(result, sh.sourceConstraint)
            sparql = report_graph.value(sc, sh.select)

            for row in data_graph.query(str(sparql)):
                if len(row) >= 2:
                    print(f"{_short(row[0])} – {_short(row[1])}: {message} ({rule_name})")
                else:
                    print(f"{' – '.join(_short(v) for v in row)}: {message} ({rule_name})")
        else:
            print(f"- {focus} violates {rule_name}: {message}")

Window rule violations found:
BTH-02: One or more expected windows are missing in this room (W1: Expected window on this room)


In [ ]:
# Debugging
from rdflib import Namespace
from rdflib.namespace import RDF
resplan = Namespace("http://resplan.org/resplan#")

inferred_windows = [
    w for w in g.subjects(RDF.type, resplan.Window)
    if 'infer#' in str(w)
]
for w in inferred_windows:
    rooms = list(g.subjects(resplan.hasWindow, w))
    print(str(w).split('#')[-1], '->', [str(r).split('#')[-1] for r in rooms])


NameError: name 'g' is not defined

In [49]:
import requests

# Inferenced output path
ttl_path = pathlib.Path("../output/inferred_resplan_ttl/plan_00010_windows_back.ttl")
ttl_data = ttl_path.read_bytes()

# configure GraphDB
repo_id = "SemanticToGeometry_01"
endpoint = f"http://localhost:7200/repositories/{repo_id}/statements"

# named graph to save
graph_iri = "http://resplan.org/graph/data/plan_00010_window_back"

# delete existing graph if available
clear_update = f"CLEAR GRAPH <{graph_iri}>"
requests.post(
    endpoint,
    data=clear_update,
    headers={"Content-Type": "application/sparql-update"}
).raise_for_status()

# Upload ttl to named graph
resp = requests.post(
    endpoint,
    params={"context": f"<{graph_iri}>"},
    data=ttl_data,
    headers={"Content-Type": "text/turtle"},
)
resp.raise_for_status()
print("Uploaded to GraphDB:", graph_iri)



Uploaded to GraphDB: http://resplan.org/graph/data/plan_00010_window_back


## Interior Wall

In [103]:
# CREATE INSTANCE INTERIOR WALL IF MISSING
from rdflib import Graph, BNode
import pathlib

# Paths
core = pathlib.Path('../ontology/core/Resplan.ttl')
input_plan = pathlib.Path('../output/imp_resplan_ttl/plan_00010_drop_interior_wall.ttl')
construct_dir = pathlib.Path('../ontology/construct')
construct_files = [
      construct_dir / 'InteriorWallConstruct.rq',
]
output_path = pathlib.Path('../output/inferred_resplan_ttl/plan_00010_walls_back.ttl')

# Merge data graph(s)
g = Graph()
for src in (core, input_plan):
    print(f'Loading: {src}')
    g.parse(src, format='turtle')
print(f'Loaded triples: {len(g)}')

# Apply CONSTRUCT queries (use g.query, not g.update)
for cq in construct_files:
    text = cq.read_text()
    print(f'Applying construct: {cq}')
    res = g.query(text)
    try:
        g += res.graph  # add constructed triples
    except Exception:
        # If result is not a Graph (older rdflib), iterate bindings
        for t in res:
            g.add(t)
print(f'Triples after construct: {len(g)}')

# Save enriched graph
output_path.parent.mkdir(parents=True, exist_ok=True)
g.serialize(output_path, format='turtle')
print(f'Saved: {output_path}')


Loading: ../ontology/core/Resplan.ttl
Loading: ../output/imp_resplan_ttl/plan_00010_drop_interior_wall.ttl
Loaded triples: 1269
Applying construct: ../ontology/construct/InteriorWallConstruct.rq
Triples after construct: 1307
Saved: ../output/inferred_resplan_ttl/plan_00010_walls_back.ttl


In [104]:
# ADJACENCY RULES CHECKER
from pyshacl import validate
from rdflib import RDF, RDFS, Graph, Namespace, Literal

def _short(node):
    node = str(node)
    return node.split('#')[-1] if '#' in node else node

data_graph = Graph().parse('../output/inferred_resplan_ttl/plan_00010_walls_back.ttl', format='turtle')
shacl_graph = Graph().parse('../ontology/rules/AdjacencyRule.shacl.ttl', format='turtle')
onto_graph = Graph().parse('../ontology/core/Resplan.ttl', format='turtle')

conforms, report_graph, _ = validate(
    data_graph,
    shacl_graph=shacl_graph,
    ont_graph=onto_graph,
    inference="rdfs",
    meta_shacl=True,
    advanced=True,
)

sh = Namespace("http://www.w3.org/ns/shacl#")

if conforms:
    print("All adjacency rules satisfied.")
else:
    print("Adjacency rule violations found:")

    handled_sparql_rules = set()

    for result in report_graph.subjects(RDF.type, sh.ValidationResult):
        raw_shape = report_graph.value(result, sh.sourceShape)

        # mapped from PropertyShape/SPARQL bnode to main NodeShape (using shacl_graph!)
        rule_node = raw_shape
        if isinstance(raw_shape, BNode):
            for ns in shacl_graph.subjects(sh.property, raw_shape):
                rule_node = ns
                break

        rule_id = _short(rule_node)
        rule_label = report_graph.value(rule_node, RDFS.label) or shacl_graph.value(rule_node, RDFS.label)
        rule_name = str(rule_label) if rule_label else rule_id

        comp = report_graph.value(result, sh.sourceConstraintComponent)
        focus = _short(report_graph.value(result, sh.focusNode))
        message_node = report_graph.value(result, sh.resultMessage)
        message = str(message_node) if isinstance(message_node, Literal) or message_node else ""

        if comp == sh.SPARQLConstraintComponent:
            if rule_node in handled_sparql_rules:
                continue  # already processed this rule
            handled_sparql_rules.add(rule_node)
            
            sc = report_graph.value(result, sh.sourceConstraint)
            sparql = report_graph.value(sc, sh.select)

            for row in data_graph.query(str(sparql)):
                if len(row) >= 2:
                    print(f"{_short(row[0])} – {_short(row[1])}: {message} ({rule_name})")
                else:
                    print(f"{' – '.join(_short(v) for v in row)}: {message} ({rule_name})")
        else:
            print(f"- {focus} violates {rule_name}: {message}")

All adjacency rules satisfied.


In [105]:
# Debugging
from rdflib import Graph, Namespace, RDF
from pathlib import Path

g = Graph().parse('output/inferred_resplan_ttl/plan_00010_walls_back.ttl', format='turtle')
resplan = Namespace('http://resplan.org/resplan#')

def short(node):
    s = str(node); return s.split('#')[-1] if '#' in s else s

rows = []
for w in g.subjects(RDF.type, resplan.InteriorWall):
    inferred = g.value(w, resplan.isInferred)
    if inferred and str(inferred).lower() == 'true':
        rooms = sorted({short(r) for r in g.subjects(resplan.boundedBy, w)})
        derived = g.value(w, resplan.derivedFrom)
        rows.append((short(w), rooms, short(derived) if derived else None))

for wid, rooms, derived in sorted(rows):
    print(f"{wid} -> rooms={rooms} derivedFrom={derived}")
print(f"Total inferred interior walls: {len(rows)}")



IW-INF-68259c0ee9cb8e86207f4a17ac7b459f74d186e6c33a1e9108132dff6f0ce453 -> rooms=['BED-01', 'BTH-01', 'BTH-02'] derivedFrom=adj-BED-01-BTH-01-IN-02
IW-INF-8d11c96b11448bb9261ba18472aa71e3331cbd549a5cb11b65cb21b2624434f0 -> rooms=['BED-01', 'LIV-01'] derivedFrom=adj-BED-01-LIV-01-IN-04
IW-INF-f47f62cfe7c5816f89706a8b6cbaa76f9a26543573f2ce1ccf907819f4345579 -> rooms=['BED-01', 'BTH-01', 'LIV-01'] derivedFrom=adj-BED-01-BTH-01-IN-03
Total inferred interior walls: 3


In [106]:
import requests

# Inferenced output path
ttl_path = pathlib.Path("../output/inferred_resplan_ttl/plan_00010_walls_back.ttl")
ttl_data = ttl_path.read_bytes()

# configure GraphDB
repo_id = "SemanticToGeometry_01"
endpoint = f"http://localhost:7200/repositories/{repo_id}/statements"

# named graph to save
graph_iri = "http://resplan.org/graph/data/plan_00010_walls_back"

# delete existing graph if available
clear_update = f"CLEAR GRAPH <{graph_iri}>"
requests.post(
    endpoint,
    data=clear_update,
    headers={"Content-Type": "application/sparql-update"}
).raise_for_status()

# Upload ttl to named graph
resp = requests.post(
    endpoint,
    params={"context": f"<{graph_iri}>"},
    data=ttl_data,
    headers={"Content-Type": "text/turtle"},
)
resp.raise_for_status()
print("Uploaded to GraphDB:", graph_iri)



Uploaded to GraphDB: http://resplan.org/graph/data/plan_00010_walls_back
